# Проверка справочника ФП/СФП

Файл на сервере: `ref_book.csv`

**Цель:** проверить, есть ли случаи, когда один и тот же номер фактора (`№`) имеет несколько разных наименований.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "../sources"
REF_FILE = f"{DATA_DIR}/ref_book.csv"

## 1. Загрузка справочника

In [ ]:
df_ref = pd.read_csv(REF_FILE, sep=";", encoding="utf-8-sig", dtype=str)

print(f"Справочник загружен: {len(df_ref):,} строк")
print(f"Колонки: {list(df_ref.columns)}")
print(f"\nУникальных номеров:      {df_ref['№'].nunique():,}")
print(f"Уникальных наименований: {df_ref['Наименование'].nunique():,}")
df_ref.head()

## 2. Общая статистика по справочнику

In [ ]:
print("=" * 55)
print("РАСПРЕДЕЛЕНИЕ ПО КЛЮЧЕВЫМ ПОЛЯМ")
print("=" * 55)

for col in ['Группа', 'Критичность', 'ЗО для ФП/СФП', 'ФП', 'СФП', 'Дефолт', 'Приостановление выдачи']:
    if col in df_ref.columns:
        print(f"\n  {col}:")
        for val, cnt in df_ref[col].value_counts().items():
            print(f"    {val:<35s} {cnt:>5,}")

## 3. Проверка: один номер — несколько наименований

In [ ]:
names_per_num = df_ref.groupby('№')['Наименование'].nunique()
dups = names_per_num[names_per_num > 1].sort_values(ascending=False)

print("=" * 65)
print("ПРОВЕРКА: один номер — несколько наименований")
print("=" * 65)
print(f"  Всего уникальных номеров:             {len(names_per_num):,}")
print(f"  Номеров с одним наименованием:        {(names_per_num == 1).sum():,}")
print(f"  Номеров с несколькими наименованиями: {len(dups):,}")

if len(dups) > 0:
    print(f"\n{'=' * 65}")
    print(f"ДЕТАЛИ (до 20 номеров):")
    print("=" * 65)
    for num in dups.index[:20]:
        subset = df_ref[df_ref['№'] == num]
        unique_names = subset['Наименование'].unique()
        groups = subset['Группа'].unique()
        print(f"\n  № {num}  ({len(subset)} строк, {len(unique_names)} наименований)")
        print(f"    Группа: {', '.join(groups)}")
        for i, name in enumerate(unique_names, 1):
            cnt = len(subset[subset['Наименование'] == name])
            print(f"    {i}. [{cnt}x] {name}")
else:
    print("\n  Дублей не обнаружено: каждый номер имеет ровно одно наименование.")

## 4. Сводная таблица дублей

In [ ]:
if len(dups) > 0:
    dup_detail = []
    for num in dups.index:
        subset = df_ref[df_ref['№'] == num]
        for name in subset['Наименование'].unique():
            row = subset[subset['Наименование'] == name].iloc[0]
            dup_detail.append({
                '№': num,
                'Наименование': name,
                'Группа': row['Группа'],
                'Критичность': row['Критичность'],
                'ФП': row['ФП'],
                'СФП': row['СФП'],
                'Дефолт': row['Дефолт'],
                'Кол-во строк': len(subset[subset['Наименование'] == name]),
            })
    df_dups = pd.DataFrame(dup_detail)
    print(f"Сводная таблица дублей ({len(df_dups)} строк):")
    display(df_dups.style.hide(axis='index'))
else:
    print("Дублей нет — таблица не нужна.")